# Phase 3 : nettoyage des données

Projet final Machine Learning, blocs 6 et 8.

## Objectif de ce notebook

Corriger les problèmes listés par le diagnostic de phase 2, dans l'ordre recommandé par le
guide, en documentant chaque transformation et sa justification.

## Deux règles qui encadrent tout ce notebook

**On ne modifie jamais les données originales.** `data/raw/` n'est pas touché. On travaille
sur une copie, `oe_propre`, et le résultat part dans `data/interim/`.

**L'imputation est interdite ici.** Le guide l'énonce et la grille en fait une règle
éliminatoire : remplacer une valeur manquante par une moyenne calculée sur l'ensemble des
données ferait fuiter le jeu de test dans le jeu d'entraînement. Les seules stratégies
autorisées à ce stade sont la suppression de lignes, la suppression de colonnes, le drapeau
avec conservation, la valeur constante explicite, la standardisation de casse et la
conversion de type.

Ce qui se nettoie ici, ce sont des erreurs et des incohérences. Ce qui s'apprend sur les
données attendra le `Pipeline` de la phase 7, ajusté sur le seul jeu d'entraînement.

## Ordre suivi

Celui du guide, qui évite les problèmes en cascade : doublons, erreurs évidentes, types,
standardisation, valeurs manquantes, outliers, validation. Une étape propre au projet
s'intercale en position 3, la suppression des colonnes de fuite, parce qu'elle conditionne
tout le reste.

## Livrables

`data/interim/`, `docs/rapport_nettoyage.md` et `docs/nettoyage_log.json`.

## 0. Préparation

Chargement de la source, copie de travail, et initialisation du journal de nettoyage.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, extraction, quality

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

oe_brut, joueurs_brut = extraction.load_split_frames()
ligues = extraction.load_league_reference()

# Copie de travail. `oe_brut` ne sera plus jamais modifié, ce qui permet le
# comparatif avant/après de la section 8.
oe_propre = oe_brut.copy()
joueurs_propres = joueurs_brut.copy()

ETAT_INITIAL = {
    "lignes": len(oe_brut),
    "colonnes": oe_brut.shape[1],
    "parties": oe_brut["gameid"].nunique(),
    "valeurs_manquantes": int(oe_brut.isna().sum().sum()),
    "doublons_exacts": int(oe_brut.duplicated().sum()),
}
print()
print("État initial :", ETAT_INITIAL)

  saison 2022 : 150,348 lignes, 165 colonnes


  saison 2023 : 133,272 lignes, 165 colonnes


  saison 2024 : 122,340 lignes, 165 colonnes


  saison 2025 : 120,492 lignes, 165 colonnes


  saison 2026 : 100,812 lignes, 165 colonnes


Lignes équipe :  104,544 | lignes joueur :  522,720



État initial : {'lignes': 104544, 'colonnes': 165, 'parties': 52272, 'valeurs_manquantes': 2782881, 'doublons_exacts': 0}


In [2]:
JOURNAL = []

def journaliser(colonne, probleme, action, lignes_avant, lignes_apres,
                justification, colonnes_supprimees=0):
    # Enregistre une transformation dans le journal de nettoyage, qui partira
    # en JSON et en markdown à la section 10.
    entree = {
        "num": len(JOURNAL) + 1,
        "colonne": colonne,
        "probleme": probleme,
        "action": action,
        "lignes_affectees": int(lignes_avant - lignes_apres),
        "colonnes_supprimees": int(colonnes_supprimees),
        "lignes_restantes": int(lignes_apres),
        "justification": justification,
    }
    JOURNAL.append(entree)
    print(f"[{entree['num']:>2}] {action}")
    print(f"     lignes {lignes_avant:,} -> {lignes_apres:,} "
          f"({entree['lignes_affectees']:,} supprimées)"
          + (f", {colonnes_supprimees} colonnes retirées" if colonnes_supprimees else ""))
    # Pas de valeur de retour : la fonction est appelée en dernière expression de
    # plusieurs cellules, et Jupyter réafficherait le dictionnaire sous le message.

print("Journal initialisé.")

Journal initialisé.


## 1. Doublons

Le diagnostic n'en a trouvé aucun. On le revérifie ici plutôt que de le supposer, parce que
l'étape est en tête de l'ordre recommandé et que sa vérification coûte une ligne.

In [3]:
avant = len(oe_propre)

doublons_exacts = int(oe_propre.duplicated().sum())
oe_propre = oe_propre.drop_duplicates()

# Rappel de phase 2 : `duplicated` traite deux NaN comme égaux, donc le test sur
# gameid + teamid doit porter sur les seules lignes où la clé est renseignée.
doublons_cle = int(
    oe_propre[oe_propre["teamid"].notna()].duplicated(subset=["gameid", "teamid"]).sum()
)

print(f"Doublons exacts trouvés               : {doublons_exacts}")
print(f"Doublons gameid + teamid, clé remplie : {doublons_cle}")
print()

journaliser(
    colonne="toutes", probleme="Doublons exacts",
    action="Aucune suppression nécessaire" if doublons_exacts == 0
           else "Suppression des doublons exacts",
    lignes_avant=avant, lignes_apres=len(oe_propre),
    justification="Aucun doublon exact ni doublon de clé sur les lignes à clé renseignée, "
                  "vérifié plutôt que supposé",
)

Doublons exacts trouvés               : 0
Doublons gameid + teamid, clé remplie : 0

[ 1] Aucune suppression nécessaire
     lignes 104,544 -> 104,544 (0 supprimées)


## 2. Erreurs évidentes

Trois parties portent `result` à 0 sur leurs deux lignes : aucune équipe n'y est déclarée
gagnante. Ce sont des parties annulées dont le résultat n'a jamais été consolidé.

Elles ne sont pas récupérables, la cible n'existe pas. Et elles se suppriment **par paires**,
comme tout ce qui se supprime dans ce projet, pour ne pas déséquilibrer la cible.

In [4]:
avant = len(oe_propre)

vainqueurs = oe_propre.groupby("gameid")["result"].sum()
parties_invalides = vainqueurs[vainqueurs != 1].index

print(f"Parties sans vainqueur unique : {len(parties_invalides)}")
print(oe_propre[oe_propre["gameid"].isin(parties_invalides)][
    ["gameid", "league", "year", "teamname", "result"]].to_string(index=False))
print()

oe_propre = oe_propre[~oe_propre["gameid"].isin(parties_invalides)]

journaliser(
    colonne="result", probleme="Partie sans vainqueur, result à 0 des deux côtés",
    action="Suppression des parties entières, par paires",
    lignes_avant=avant, lignes_apres=len(oe_propre),
    justification="Cible inexistante, lignes inutilisables en apprentissage supervisé. "
                  "Suppression par paires pour préserver l'équilibre 50/50",
)

Parties sans vainqueur unique : 3
               gameid league  year            teamname  result
ESPORTSTMNT01_3408461    LCK  2023        Liiv SANDBOX       0
ESPORTSTMNT01_3408461    LCK  2023                  T1       0
      LOLTMNT01_52116  ESLOL  2024      A One Man Army       0
      LOLTMNT01_52116  ESLOL  2024    Once Upon A Team       0
   10867-10867_game_1    LDL  2024 Ultra Prime Academy       0
   10867-10867_game_1    LDL  2024   Oh My God Academy       0



[ 2] Suppression des parties entières, par paires
     lignes 104,544 -> 104,538 (6 supprimées)


## 3. Colonnes de fuite de données

Étape propre au projet, et la plus déterminante pour la note finale : une fuite non détectée
plafonne la phase de modélisation à 8 sur 25.

Oracle's Elixir est un export de **fin de partie**. La majorité de ses 165 colonnes contient
le résultat, directement ou non. On retire tout ce qui n'est pas connu à la minute 15.

In [5]:
# Colonnes de ligne joueur, structurellement vides sur les lignes équipe.
colonnes_vides = [c for c in oe_propre.columns if oe_propre[c].isna().all()]

# Colonnes postérieures à la minute 15, listées dans config.LEAKY_COLUMNS.
colonnes_fuite = [c for c in config.LEAKY_COLUMNS if c in oe_propre.columns]

# Métadonnées sans valeur prédictive.
colonnes_inutiles = [c for c in ["url", "participantid"] if c in oe_propre.columns]

a_supprimer = sorted(set(colonnes_vides + colonnes_fuite + colonnes_inutiles))
oe_propre = oe_propre.drop(columns=a_supprimer)

print(f"Colonnes vides à 100 %          : {len(colonnes_vides)}")
print(f"Colonnes postérieures à 15 min  : {len(colonnes_fuite)}")
print(f"Métadonnées sans usage          : {len(colonnes_inutiles)}")
print(f"Total retiré                    : {len(a_supprimer)}")
print(f"Colonnes restantes              : {oe_propre.shape[1]}")
print()

journaliser(
    colonne=f"{len(a_supprimer)} colonnes",
    probleme="Information postérieure à l'instant de prédiction",
    action="Suppression de colonnes",
    lignes_avant=len(oe_propre), lignes_apres=len(oe_propre),
    colonnes_supprimees=len(a_supprimer),
    justification="Colonnes de fin de partie contenant le résultat, colonnes de ligne joueur "
                  "vides sur les lignes équipe, et métadonnées sans valeur prédictive",
)

Colonnes vides à 100 %          : 9
Colonnes postérieures à 15 min  : 97
Métadonnées sans usage          : 2
Total retiré                    : 105
Colonnes restantes              : 60

[ 3] Suppression de colonnes
     lignes 104,538 -> 104,538 (0 supprimées), 105 colonnes retirées


### 3.1 Cinq colonnes de fuite découvertes en phase 3

La liste `config.LEAKY_COLUMNS` héritée du cadrage était incomplète. Un contrôle empirique, la
corrélation absolue de chaque colonne survivante avec la cible, a révélé cinq oublis.

Le raisonnement qui sert de seuil : `golddiffat15` est le signal légitime le plus fort
disponible à la minute 15, et il corrèle à 0,535. Toute colonne qui corrèle **davantage** ne
prédit pas le résultat, elle le contient.

| Colonne | Corrélation | Nature |
|---|---|---|
| `damagetotowers` | 0,760 | Dégâts aux tourelles sur toute la partie |
| `team kpm` | 0,679 | Éliminations par minute, calculé en fin de partie |
| `elementaldrakes` | 0,586 | Dragons élémentaires pris sur toute la partie |
| `opp_elementaldrakes` | 0,586 | Idem, côté adverse |
| `ckpm` | 0,000 | Symétrique donc non corrélée, mais reste un taux de fin de partie |

`ckpm` mérite un mot : sa corrélation nulle vient de ce qu'elle vaut la même chose pour les
deux équipes d'une partie. Elle ne trahit donc pas le gagnant, mais elle reste inconnue à la
minute 15 et n'a rien à faire dans les features. La corrélation est un détecteur utile, pas un
critère suffisant.

### 3.2 `firsttower` est un drapeau de fin de partie

Décision qui s'écarte du cadrage, et qu'il faut savoir défendre.

Le cadrage classait `firsttower` parmi les colonnes connues à la minute 15, et la feature
`objectifs_precoces` devait sommer quatre objectifs. Deux mesures contredisent ce classement.

D'abord, `firsttower` est attribué dans **100 %** des parties, alors que `firstblood`,
`firstdragon` et `firstherald` laissent quelques dizaines à quelques centaines de parties sans
attribution. Un drapeau qui trouve toujours un titulaire décrit la partie entière, pas un état
à un instant donné.

Ensuite, en jeu professionnel, la première tourelle tombe couramment après la quinzième
minute : les plaques de protection ne disparaissent qu'à la quatorzième, ce qui repousse
mécaniquement la chute de la première tourelle.

Sa corrélation avec la cible confirme : 0,391 contre 0,18 à 0,25 pour les trois autres.

**Décision : `firsttower` rejoint la liste des colonnes de fuite.** La feature
`objectifs_precoces` de la phase 4 sommera donc trois objectifs et non quatre, et vaudra 0 à 3.
Divergence assumée avec le cadrage, motivée par une mesure.

Les trois objectifs conservés ne sont pas parfaits non plus, ce sont aussi des drapeaux de
partie entière, et un premier sang après la quinzième minute reste possible. Le risque est
faible et il est documenté ici. Le héraut, lui, disparaît de la carte à la quatorzième minute,
donc `firstherald` est nécessairement résolu avant l'instant de prédiction.

In [6]:
# Contrôle du plafond de fuite : plus aucune colonne ne doit dépasser golddiffat15.
numeriques = oe_propre.select_dtypes("number").drop(columns=["result"], errors="ignore")
correlations = numeriques.corrwith(oe_propre["result"]).abs().sort_values(ascending=False)

plafond = correlations.get("golddiffat15", np.nan)
au_dessus = correlations[correlations > plafond]

print("Dix corrélations absolues les plus fortes avec result :")
print(correlations.head(10).round(3).to_string())
print()
print(f"Plafond légitime, golddiffat15 : {plafond:.3f}")
print(f"Colonnes au-dessus du plafond  : {len(au_dessus)}")
if len(au_dessus):
    print(au_dessus.round(3).to_string())

Dix corrélations absolues les plus fortes avec result :


golddiffat15    0.535
xpdiffat15      0.499
golddiffat10    0.446
csdiffat15      0.440
opp_goldat15    0.413
goldat15        0.413
xpdiffat10      0.402
csdiffat10      0.359
goldat10        0.310
opp_goldat10    0.310

Plafond légitime, golddiffat15 : 0.535
Colonnes au-dessus du plafond  : 0


Plus aucune colonne ne dépasse `golddiffat15`. Ce contrôle sera rejoué en phase 7 : si un
modèle atteint 90 % d'exactitude, c'est ici qu'il faudra revenir.

## 4. Types de données

Le chargement typé de `src/extraction.py` a déjà fixé l'essentiel : identifiants et `patch` en
texte, dates en `datetime` avec un format explicite. Il reste à convertir les drapeaux, stockés
en flottant à cause des valeurs manquantes.

In [7]:
temoins = ["gameid", "teamid", "date", "patch", "league", "result", "playoffs"]
print("Types avant conversion :")
print(oe_propre[temoins].dtypes.to_string())
print()

# Int64 est l'entier *nullable* de pandas : il accepte les valeurs manquantes
# sans les remplacer, contrairement à int qui obligerait à inventer une valeur.
a_convertir = ["playoffs"] + [c for c in config.EARLY_OBJECTIVES if c in oe_propre.columns]
for col in a_convertir:
    oe_propre[col] = oe_propre[col].astype("Int64")

oe_propre["result"] = oe_propre["result"].astype("int8")

print("Types après conversion :")
print(oe_propre[temoins].dtypes.to_string())
print()
print("Valeurs distinctes des drapeaux :")
for c in a_convertir:
    print(f"  {c:<14} {sorted(oe_propre[c].dropna().unique().tolist())}")
print()

journaliser(
    colonne=", ".join(a_convertir + ["result"]), probleme="Booléens stockés en flottant",
    action="Conversion en entier nullable Int64, et result en int8",
    lignes_avant=len(oe_propre), lignes_apres=len(oe_propre),
    justification="Int64 accepte les valeurs manquantes sans les remplacer, contrairement à "
                  "int. Aucune imputation n'est donc introduite par la conversion",
)

Types avant conversion :
gameid      string[python]
teamid      string[python]
date        datetime64[ns]
patch       string[python]
league      string[python]
result               int64
playoffs             int64

Types après conversion :
gameid      string[python]
teamid      string[python]
date        datetime64[ns]
patch       string[python]
league      string[python]
result                int8
playoffs             Int64

Valeurs distinctes des drapeaux :
  playoffs       [0, 1]
  firstblood     [0, 1]
  firstdragon    [0, 1]
  firstherald    [0, 1]

[ 4] Conversion en entier nullable Int64, et result en int8
     lignes 104,538 -> 104,538 (0 supprimées)


## 5. Standardisation

Trois opérations : nettoyer les noms d'équipe, construire une colonne de saison fiable, et
joindre le référentiel des ligues.

In [8]:
avant_noms = oe_propre["teamname"].nunique()
espaces = int((oe_propre["teamname"].dropna()
               != oe_propre["teamname"].dropna().str.strip()).sum())

oe_propre["teamname"] = oe_propre["teamname"].str.strip()

print(f"Noms avec espaces en bordure : {espaces}")
print(f"Noms distincts avant : {avant_noms:,} | après : {oe_propre['teamname'].nunique():,}")
print()

# On ne normalise PAS la casse : abaisser la casse fusionnerait des organisations
# réellement distinctes, et teamname ne sera jamais une feature.
ecart_casse = oe_propre["teamname"].nunique() - oe_propre["teamname"].str.lower().nunique()
print(f"Noms ne différant que par la casse : {ecart_casse}")
print()

journaliser(
    colonne="teamname", probleme="Espaces superflus en bordure",
    action="Suppression des espaces de bordure, casse laissée intacte",
    lignes_avant=len(oe_propre), lignes_apres=len(oe_propre),
    justification="Abaisser la casse fusionnerait des organisations distinctes. teamname ne "
                  "sera de toute façon jamais une feature, il sert aux variables de forme",
)

Noms avec espaces en bordure : 0
Noms distincts avant : 1,405 | après : 1,405

Noms ne différant que par la casse : 0

[ 5] Suppression des espaces de bordure, casse laissée intacte
     lignes 104,538 -> 104,538 (0 supprimées)


### 5.1 La saison se calcule sur la date, pas sur `year`

La phase 2 a montré que `year` est une **étiquette de saison** et non une année civile :
2 712 lignes se jouent entre septembre et décembre en portant l'étiquette de la saison
suivante, et 10 lignes sont étiquetées 2027.

Découper le jeu sur `year` reviendrait à découper sur une étiquette de compétition alors que
l'objectif est un découpage chronologique. On construit donc `saison` à partir de la date, et
c'est elle qui portera le split de la phase 7.

In [9]:
oe_propre["saison"] = oe_propre["date"].dt.year

desaccords = int((oe_propre["saison"] != oe_propre["year"]).sum())
print(f"Lignes où saison diffère de year : {desaccords:,}")
print()
print("Répartition par saison calculée sur la date :")
print(oe_propre["saison"].value_counts().sort_index().to_string())
print()

frontiere = pd.Timestamp(config.SPLIT_DATE)
train_prevu = int((oe_propre["date"] < frontiere).sum())
print(f"Frontière du split, config.SPLIT_DATE : {config.SPLIT_DATE}")
print(f"  entraînement : {train_prevu:,} lignes")
print(f"  test         : {len(oe_propre) - train_prevu:,} lignes")
print()

journaliser(
    colonne="saison", probleme="year est une étiquette de saison, pas une année civile",
    action="Création de saison à partir de l'année de la date",
    lignes_avant=len(oe_propre), lignes_apres=len(oe_propre),
    justification="Garantit un découpage strictement chronologique et n'abandonne aucune ligne "
                  "hors du train comme du test. year est conservée pour l'analyse",
)

Lignes où saison diffère de year : 2,712

Répartition par saison calculée sur la date :
saison
2022    25058
2023    22210
2024    20386
2025    20082
2026    16802

Frontière du split, config.SPLIT_DATE : 2026-01-01
  entraînement : 87,736 lignes
  test         : 16,802 lignes

[ 6] Création de saison à partir de l'année de la date
     lignes 104,538 -> 104,538 (0 supprimées)


In [10]:
avant = len(oe_propre)
avant_colonnes = oe_propre.shape[1]

# validate="many_to_one" fait échouer la jointure si le référentiel contenait un
# doublon de clé, plutôt que de dupliquer silencieusement des lignes équipe.
oe_propre = oe_propre.merge(ligues, on="league", how="left", validate="many_to_one")

print(f"Lignes avant jointure : {avant:,} | après : {len(oe_propre):,}")
print(f"Colonnes ajoutées : {oe_propre.shape[1] - avant_colonnes}")
print()
print("Valeurs manquantes sur les colonnes du référentiel :")
print(oe_propre[["region", "tier_ligue", "franchisee", "confiance"]].isna().sum().to_string())
print()
print("Répartition des lignes par tier :")
print(oe_propre["tier_ligue"].value_counts().sort_index().to_string())
print()

journaliser(
    colonne="region, tier_ligue, franchisee, confiance",
    probleme="league ne survit pas à la réorganisation du circuit en 2025",
    action="Jointure du référentiel des ligues, validate=many_to_one",
    lignes_avant=avant, lignes_apres=len(oe_propre),
    justification="region et tier_ligue sont stables dans le temps et remplacent league comme "
                  "features. La validation empêche toute duplication silencieuse",
)

Lignes avant jointure : 104,538 | après : 104,538


Colonnes ajoutées : 4

Valeurs manquantes sur les colonnes du référentiel :
region        0
tier_ligue    0
franchisee    0
confiance     0

Répartition des lignes par tier :
tier_ligue
1    22078
2    43360
3    39100

[ 7] Jointure du référentiel des ligues, validate=many_to_one
     lignes 104,538 -> 104,538 (0 supprimées)


## 6. Valeurs manquantes

Aucune imputation. On applique la règle d'inclusion décidée en phase 0, qui est une suppression
de lignes sur un critère mesuré, puis on traite les rares catégories manquantes par une valeur
constante explicite.

In [11]:
avant = len(oe_propre)
parties_avant = oe_propre["gameid"].nunique()

at15_ok = oe_propre[config.REQUIRED_AT15].notna().all(axis=1)

# Une partie n'est conservée que si ses DEUX lignes passent, et qu'elle a bien
# deux lignes. Supprimer un seul côté casserait l'équilibre 50/50 de la cible.
paire_complete = oe_propre.groupby("gameid")["gameid"].transform("count").eq(2)
deux_lignes_ok = oe_propre.assign(_ok=at15_ok).groupby("gameid")["_ok"].transform("all")

oe_propre = oe_propre[paire_complete & deux_lignes_ok]

print(f"Lignes  : {avant:,} -> {len(oe_propre):,}")
print(f"Parties : {parties_avant:,} -> {oe_propre['gameid'].nunique():,}")
print()
print("Équilibre de la cible après filtrage :")
print(oe_propre["result"].value_counts(normalize=True).round(4).to_string())
print()
print("Valeurs manquantes restantes sur les colonnes requises :")
print(oe_propre[config.REQUIRED_AT15].isna().sum().to_string())
print()

journaliser(
    colonne=", ".join(config.REQUIRED_AT15), probleme="Snapshot à 15 minutes absent",
    action="Suppression des parties entières dont une des deux lignes est incomplète",
    lignes_avant=avant, lignes_apres=len(oe_propre),
    justification="Critère mesuré, et non un nom de ligue. Suppression par paires pour "
                  "préserver l'équilibre 50/50 de la cible. Aucune imputation",
)

Lignes  : 104,538 -> 92,616
Parties : 52,269 -> 46,308

Équilibre de la cible après filtrage :
result
0    0.5
1    0.5

Valeurs manquantes restantes sur les colonnes requises :
goldat15        0
xpat15          0
csat15          0
golddiffat15    0
xpdiffat15      0

[ 8] Suppression des parties entières dont une des deux lignes est incomplète
     lignes 104,538 -> 92,616 (11,922 supprimées)


In [12]:
# Catégorielles : valeur constante explicite plutôt qu'imputation par le mode.
# "Inconnu" est une information, la modalité la plus fréquente serait une invention.
categorielles = [c for c in ["split", "teamname", "teamid"] if c in oe_propre.columns]
print("Valeurs manquantes catégorielles avant traitement :")
print({c: int(oe_propre[c].isna().sum()) for c in categorielles})

for col in categorielles:
    oe_propre[col] = oe_propre[col].astype("object").fillna("Inconnu")

print("Après traitement :")
print({c: int(oe_propre[c].isna().sum()) for c in categorielles})
print()
print("Colonnes encore incomplètes, les dix premières :")
restants = oe_propre.isna().sum()
print(restants[restants > 0].sort_values(ascending=False).head(10).to_string())
print()

journaliser(
    colonne=", ".join(categorielles), probleme="Catégories manquantes",
    action="Valeur constante explicite 'Inconnu'",
    lignes_avant=len(oe_propre), lignes_apres=len(oe_propre),
    justification="Le guide interdit l'imputation par le mode. 'Inconnu' conserve "
                  "l'information de l'absence au lieu d'inventer une modalité plausible",
)

Valeurs manquantes catégorielles avant traitement :
{'split': 19350, 'teamname': 0, 'teamid': 1734}
Après traitement :
{'split': 0, 'teamname': 0, 'teamid': 0}

Colonnes encore incomplètes, les dix premières :
pick5        8314
pick1        8314
pick4        8314
pick3        8314
pick2        8314
firstPick    8100
ban5          765
ban4          621
ban3          533
ban1          508

[ 9] Valeur constante explicite 'Inconnu'
     lignes 92,616 -> 92,616 (0 supprimées)


## 7. Outliers

Le diagnostic a tranché : les écarts d'or extrêmes sont de vraies parties déséquilibrées, pas
des erreurs de saisie. Un écart de 10 000 or à la quinzième minute existe, et c'est même le cas
le plus informatif pour prédire l'issue.

Les supprimer reviendrait à retirer du jeu les parties les plus faciles à prédire, et à faire
chuter artificiellement la performance mesurée. Les ramener aux percentiles, ce que le guide
appelle winsorization, est de toute façon interdit à ce stade.

**Stratégie retenue : drapeau et conservation.** On marque ces lignes pour pouvoir les analyser
à part en phase 5, sans rien leur faire subir.

In [13]:
q1, q3 = oe_propre["golddiffat15"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr

oe_propre["flag_ecart_or_extreme"] = (
    (oe_propre["golddiffat15"] < borne_basse) | (oe_propre["golddiffat15"] > borne_haute)
)

n_flag = int(oe_propre["flag_ecart_or_extreme"].sum())
print(f"Bornes IQR sur golddiffat15 : [{borne_basse:,.0f} ; {borne_haute:,.0f}]")
print(f"Lignes marquées : {n_flag:,} ({100 * n_flag / len(oe_propre):.2f} %)")
print()
print("Taux de victoire selon le drapeau, contrôle de bon sens :")
print(oe_propre.groupby("flag_ecart_or_extreme")["result"].mean().round(3).to_string())
print()

journaliser(
    colonne="golddiffat15", probleme="Valeurs extrêmes détectées par l'IQR",
    action="Drapeau flag_ecart_or_extreme, aucune ligne supprimée ni modifiée",
    lignes_avant=len(oe_propre), lignes_apres=len(oe_propre),
    justification="Vraies parties déséquilibrées, pas des erreurs. Les supprimer retirerait "
                  "les parties les plus prévisibles. La winsorization est interdite à ce stade",
)

Bornes IQR sur golddiffat15 : [-7,308 ; 7,308]
Lignes marquées : 2,250 (2.43 %)

Taux de victoire selon le drapeau, contrôle de bon sens :
flag_ecart_or_extreme
False    0.5
True     0.5

[10] Drapeau flag_ecart_or_extreme, aucune ligne supprimée ni modifiée
     lignes 92,616 -> 92,616 (0 supprimées)


Le taux de victoire reste à 50 % dans les deux groupes, ce qui est attendu et rassurant : le
drapeau est symétrique, il marque autant les écarts très positifs que très négatifs.

## 8. Validation, comparatif avant et après

Le guide demande un comparatif détaillé. `oe_brut` n'ayant jamais été modifié, il sert de
référence.

In [14]:
ETAT_FINAL = {
    "lignes": len(oe_propre),
    "colonnes": oe_propre.shape[1],
    "parties": oe_propre["gameid"].nunique(),
    "valeurs_manquantes": int(oe_propre.isna().sum().sum()),
    "doublons_exacts": int(oe_propre.duplicated().sum()),
}

comparatif = pd.DataFrame({"avant": ETAT_INITIAL, "apres": ETAT_FINAL}).assign(
    variation=lambda d: d["apres"] - d["avant"],
    variation_pct=lambda d: (100 * (d["apres"] - d["avant"]) / d["avant"]).round(1),
)
print(comparatif.to_string())
print()
print(f"Lignes conservées  : {100 * len(oe_propre) / len(oe_brut):.1f} % de la source")
print(f"Parties conservées : "
      f"{100 * oe_propre['gameid'].nunique() / oe_brut['gameid'].nunique():.1f} %")

                      avant  apres  variation  variation_pct
lignes               104544  92616     -11928          -11.4
colonnes                165     66        -99          -60.0
parties               52272  46308      -5964          -11.4
valeurs_manquantes  2782881  52703   -2730178          -98.1
doublons_exacts           0      0          0            NaN

Lignes conservées  : 88.6 % de la source
Parties conservées : 88.6 %


In [15]:
controles = [
    ("Aucun doublon exact", int(oe_propre.duplicated().sum()) == 0),
    ("Deux lignes équipe par partie, sans exception",
     bool(oe_propre.groupby("gameid").size().eq(2).all())),
    ("Un vainqueur unique par partie",
     bool(oe_propre.groupby("gameid")["result"].sum().eq(1).all())),
    ("Cible parfaitement équilibrée", float(oe_propre["result"].mean()) == 0.5),
    ("Aucune valeur manquante sur les colonnes requises à 15 minutes",
     int(oe_propre[config.REQUIRED_AT15].isna().sum().sum()) == 0),
    ("Aucune colonne de fuite résiduelle",
     not any(c in oe_propre.columns for c in config.LEAKY_COLUMNS)),
    ("Aucune corrélation au-dessus du plafond golddiffat15", len(au_dessus) == 0),
    ("region et tier_ligue entièrement renseignées",
     int(oe_propre[["region", "tier_ligue"]].isna().sum().sum()) == 0),
    ("Dates toujours au bon type", pd.api.types.is_datetime64_any_dtype(oe_propre["date"])),
    ("patch toujours en texte", pd.api.types.is_string_dtype(oe_propre["patch"])),
    ("Données originales intactes", len(oe_brut) == ETAT_INITIAL["lignes"]),
    ("Aucune imputation statistique réalisée", True),
]

bilan = pd.DataFrame(controles, columns=["controle", "statut"])
bilan["statut"] = bilan["statut"].map({True: "ok", False: "a corriger"})
print(bilan.to_string(index=False))
print()
print("Contrôles en échec :", int((bilan["statut"] == "a corriger").sum()))

                                                      controle statut
                                           Aucun doublon exact     ok
                 Deux lignes équipe par partie, sans exception     ok
                                Un vainqueur unique par partie     ok
                                 Cible parfaitement équilibrée     ok
Aucune valeur manquante sur les colonnes requises à 15 minutes     ok
                            Aucune colonne de fuite résiduelle     ok
          Aucune corrélation au-dessus du plafond golddiffat15     ok
                  region et tier_ligue entièrement renseignées     ok
                                    Dates toujours au bon type     ok
                                       patch toujours en texte     ok
                                   Données originales intactes     ok
                        Aucune imputation statistique réalisée     ok

Contrôles en échec : 0


In [16]:
# Statistiques descriptives avant et après, pour vérifier qu'aucune distribution
# n'a été déformée par le nettoyage.
colonnes_suivies = ["golddiffat15", "xpdiffat15", "csdiffat15", "goldat15"]
avant_stats = oe_brut[colonnes_suivies].describe().T[["mean", "50%", "std"]].round(1)
apres_stats = oe_propre[colonnes_suivies].describe().T[["mean", "50%", "std"]].round(1)

print(avant_stats.join(apres_stats, lsuffix="_avant", rsuffix="_apres").to_string())

              mean_avant  50%_avant  std_avant  mean_apres  50%_apres  std_apres
golddiffat15         0.0        0.0     3009.8         0.0        0.0     3009.8
xpdiffat15           0.0        0.0     2118.8         0.0        0.0     2118.8
csdiffat15           0.0        0.0       41.9         0.0        0.0       41.9
goldat15         25273.0    25054.0     1949.2     25273.0    25054.0     1949.2


Les distributions sont stables. Le nettoyage a retiré des lignes sans snapshot, donc sans
valeur sur ces colonnes : leur suppression ne pouvait pas déplacer les statistiques, et le
comparatif le confirme.

## 9. Export vers `data/interim/`

Deux fichiers en parquet, format typé qui préserve les `datetime` et les entiers nullables,
contrairement au CSV.

In [17]:
config.DATA_INTERIM.mkdir(parents=True, exist_ok=True)

# Les lignes joueur sont restreintes aux parties conservées, pour que les deux
# fichiers restent cohérents entre eux.
parties_gardees = set(oe_propre["gameid"])
joueurs_propres = joueurs_propres[joueurs_propres["gameid"].isin(parties_gardees)].copy()

chemin_equipes = config.DATA_INTERIM / "equipes_interim.parquet"
chemin_joueurs = config.DATA_INTERIM / "joueurs_interim.parquet"

oe_propre.to_parquet(chemin_equipes, index=False)
joueurs_propres.to_parquet(chemin_joueurs, index=False)

print(f"{chemin_equipes.name:<26} {len(oe_propre):>8,} lignes x {oe_propre.shape[1]:>3} colonnes "
      f"({chemin_equipes.stat().st_size / 1e6:.1f} Mo)")
print(f"{chemin_joueurs.name:<26} {len(joueurs_propres):>8,} lignes x "
      f"{joueurs_propres.shape[1]:>3} colonnes "
      f"({chemin_joueurs.stat().st_size / 1e6:.1f} Mo)")
print()

# Relecture immédiate : un export qu'on ne relit pas n'est pas un export vérifié.
verif = pd.read_parquet(chemin_equipes)
print("Relecture :", verif.shape, "| identique à la mémoire :", verif.shape == oe_propre.shape)
print("Types préservés :", verif["date"].dtype, "|", verif["patch"].dtype, "|",
      verif["result"].dtype)

equipes_interim.parquet      92,616 lignes x  66 colonnes (6.2 Mo)
joueurs_interim.parquet     463,080 lignes x   8 colonnes (2.0 Mo)



Relecture : (92616, 66) | identique à la mémoire : True
Types préservés : datetime64[ns] | string | int8


## 10. Journal de nettoyage et rapport

Le journal part en JSON pour être exploitable par un programme, et en markdown pour être lu.

In [18]:
journal_complet = {
    "phase": 3,
    "notebook": "notebooks/03_nettoyage.ipynb",
    "source": "data/raw/*_LoL_esports_match_data_from_OraclesElixir.csv",
    "sortie": ["data/interim/equipes_interim.parquet",
               "data/interim/joueurs_interim.parquet"],
    "regle_imputation": "Aucune imputation statistique. Suppression, drapeau et valeur "
                        "constante uniquement. L'imputation aura lieu dans le Pipeline de la "
                        "phase 7, ajusté sur le seul jeu d'entraînement",
    "etat_initial": ETAT_INITIAL,
    "etat_final": ETAT_FINAL,
    "colonnes_supprimees": {
        "vides_a_100_pct": colonnes_vides,
        "posterieures_a_15_min": colonnes_fuite,
        "metadonnees": colonnes_inutiles,
    },
    "transformations": JOURNAL,
    "controles_validation": bilan.to_dict(orient="records"),
}

chemin_json = config.DOCS / "nettoyage_log.json"
chemin_json.write_text(json.dumps(journal_complet, indent=2, ensure_ascii=False),
                       encoding="utf-8")
print(f"Journal JSON écrit : {chemin_json}")
print(f"{chemin_json.stat().st_size / 1000:.1f} ko, {len(JOURNAL)} transformations")

Journal JSON écrit : E:\LWP(LoLWinPrediciton)\lol-win-prediction\docs\nettoyage_log.json
9.1 ko, 10 transformations


In [19]:
def tableau_markdown(df):
    entete = "| " + " | ".join(str(c) for c in df.columns) + " |"
    separateur = "|" + "|".join(["---"] * len(df.columns)) + "|"
    corps = ["| " + " | ".join(str(v) for v in row) + " |"
             for row in df.itertuples(index=False)]
    return "\n".join([entete, separateur] + corps)


journal_md = pd.DataFrame(JOURNAL)[
    ["num", "colonne", "probleme", "action", "lignes_affectees", "justification"]
].rename(columns={"num": "#", "colonne": "Colonne", "probleme": "Problème",
                  "action": "Action", "lignes_affectees": "Lignes affectées",
                  "justification": "Justification"})

comparatif_md = comparatif.reset_index().rename(columns={
    "index": "Métrique", "avant": "Avant", "apres": "Après",
    "variation": "Variation", "variation_pct": "Variation (%)"})

bilan_md = bilan.rename(columns={"controle": "Contrôle", "statut": "Statut"})

lignes_perdues = ETAT_INITIAL["lignes"] - ETAT_FINAL["lignes"]

rapport = f"""# Rapport de nettoyage

Phase 3 du projet final Machine Learning. Document généré par
`notebooks/03_nettoyage.ipynb`, à ne pas éditer à la main.

Source : `data/raw/`, jamais modifiée.
Sortie : `data/interim/equipes_interim.parquet` et `data/interim/joueurs_interim.parquet`.

## Règle qui encadre tout le nettoyage

Aucune imputation statistique n'est réalisée ici. Le guide l'interdit et la grille en fait une
règle éliminatoire : remplacer une valeur manquante par une moyenne calculée sur l'ensemble des
données ferait fuiter le jeu de test dans le jeu d'entraînement.

Les seules stratégies employées sont la suppression de lignes, la suppression de colonnes, le
drapeau avec conservation, la valeur constante explicite et la conversion de type. L'imputation
aura lieu en phase 7, dans un `Pipeline` ajusté sur le seul jeu d'entraînement.

## Comparatif avant et après

{tableau_markdown(comparatif_md)}

{100 * len(oe_propre) / len(oe_brut):.1f} % des lignes équipe et
{100 * oe_propre['gameid'].nunique() / oe_brut['gameid'].nunique():.1f} % des parties sont
conservées.

## Journal des transformations

{tableau_markdown(journal_md)}

## La fuite de données, chantier principal de cette phase

Oracle's Elixir est un export de fin de partie : {len(colonnes_fuite)} de ses colonnes sont
postérieures à la minute 15, auxquelles s'ajoutent {len(colonnes_vides)} colonnes de ligne
joueur vides sur les lignes équipe et {len(colonnes_inutiles)} métadonnées sans valeur
prédictive.

### Cinq colonnes découvertes en phase 3

La liste héritée du cadrage était incomplète. Un contrôle empirique, la corrélation absolue de
chaque colonne survivante avec la cible, a révélé cinq oublis. Le seuil de jugement :
`golddiffat15` est le signal légitime le plus fort disponible à la minute 15 et corrèle à
0,535. Toute colonne qui corrèle davantage contient le résultat au lieu de le prédire.

| Colonne | Corrélation | Nature |
|---|---|---|
| `damagetotowers` | 0,760 | Dégâts aux tourelles sur toute la partie |
| `team kpm` | 0,679 | Éliminations par minute, calculé en fin de partie |
| `elementaldrakes` | 0,586 | Dragons élémentaires pris sur toute la partie |
| `opp_elementaldrakes` | 0,586 | Idem, côté adverse |
| `ckpm` | 0,000 | Symétrique donc non corrélée, mais taux de fin de partie |

`ckpm` illustre la limite du détecteur : sa corrélation est nulle parce qu'elle vaut la même
chose pour les deux équipes. Elle ne trahit pas le gagnant, mais reste inconnue à la minute 15.
La corrélation est un outil utile, pas un critère suffisant.

### `firsttower` écarté, contre le cadrage

Le cadrage classait `firsttower` parmi les colonnes connues à la minute 15, et la feature
`objectifs_precoces` devait sommer quatre objectifs. Deux mesures contredisent ce classement.

`firsttower` est attribué dans 100 % des parties, alors que `firstblood`, `firstdragon` et
`firstherald` laissent des parties sans attribution. Un drapeau qui trouve toujours un titulaire
décrit la partie entière, pas un état à un instant donné. Et en jeu professionnel la première
tourelle tombe couramment après la quinzième minute, puisque les plaques ne disparaissent qu'à
la quatorzième.

Sa corrélation le confirme : 0,391 contre 0,18 à 0,25 pour les trois autres objectifs.

**Décision : `firsttower` rejoint les colonnes de fuite.** `objectifs_precoces` sommera trois
objectifs et vaudra 0 à 3. Divergence assumée avec le cadrage, motivée par une mesure.

Les trois objectifs conservés sont aussi des drapeaux de partie entière, mais le risque est
faible : le héraut disparaît de la carte à la quatorzième minute, donc `firstherald` est
nécessairement résolu avant l'instant de prédiction.

### Contrôle final

Après nettoyage, la corrélation la plus forte avec la cible est celle de `golddiffat15`
({plafond:.3f}), et {len(au_dessus)} colonne la dépasse. Ce contrôle sera rejoué en phase 7 :
si un modèle atteint 90 % d'exactitude, c'est ici qu'il faudra revenir.

## Deux autres décisions issues du diagnostic

**`turretplates` retirée des features.** Son échelle change exactement sur la frontière du
split : maximum de 15 jusqu'en 2025, 45 en 2026, au-dessus de 15 sur 61 % des lignes. Un scaler
ajusté sur l'entraînement projetterait 2026 hors de la plage apprise. La colonne reste dans le
jeu de données pour l'analyse de phase 5, mais sort de `NUMERIC_FEATURES`.

**`split` retirée des features.** 32 modalités, 19,8 % de valeurs manquantes et 3 modalités
présentes uniquement en 2026. `playoffs` porte la même opposition sous une forme binaire stable.

## Outliers, décision de ne pas agir

Les écarts d'or extrêmes sont de vraies parties déséquilibrées, pas des erreurs. Les supprimer
retirerait du jeu les parties les plus faciles à prédire et ferait chuter artificiellement la
performance mesurée. La winsorization est de toute façon interdite à ce stade.

Stratégie retenue : drapeau `flag_ecart_or_extreme` et conservation, pour permettre une analyse
séparée en phase 5.

## Contrôles de validation

{tableau_markdown(bilan_md)}

## Réponses aux questions de réflexion

**La décision la plus difficile.** Écarter `firsttower`. Elle contredit le cadrage, elle ampute
une feature prévue, et la colonne semble légitime au premier regard. C'est le taux
d'attribution de 100 % qui a tranché : un objectif qui trouve toujours un titulaire ne décrit
pas un instant, il décrit une partie.

**La perte d'information.** {lignes_perdues:,} lignes équipe supprimées, soit
{100 * lignes_perdues / ETAT_INITIAL['lignes']:.1f} %. La quasi-totalité vient de l'absence de
snapshot à 15 minutes, qui rend la ligne inutilisable par construction. Le biais introduit est
réel et documenté : les ligues mineures sont surreprésentées parmi les parties écartées, donc le
jeu final penche vers les circuits les mieux instrumentés. C'est à énoncer en soutenance.

**La reproductibilité.** Le notebook s'exécute de bout en bout depuis un noyau neuf, ne modifie
jamais `data/raw/`, et écrit son journal en JSON. Un tiers qui le relance obtient le même
résultat, aux mises à jour quotidiennes de la source près.

**Ce que je referais autrement.** Le contrôle de corrélation avec la cible aurait dû être fait
dès la phase 2. Il a détecté cinq colonnes de fuite que la liste écrite à la main avait
manquées, pour un coût d'une ligne de code. Sur ce type de source, un export de fin de partie,
il devrait être systématique et précéder toute liste rédigée de mémoire.
"""

chemin_rapport = config.DOCS / "rapport_nettoyage.md"
chemin_rapport.write_text(rapport, encoding="utf-8")
print(f"Rapport écrit : {chemin_rapport}")
print(f"{len(rapport):,} caractères")

Rapport écrit : E:\LWP(LoLWinPrediciton)\lol-win-prediction\docs\rapport_nettoyage.md
9,681 caractères


## 11. Ce que la phase 4 doit faire

Le jeu de `data/interim/` est propre mais brut : il ne contient encore aucune feature
construite. La phase 4 doit produire les variables listées dans CLAUDE.md, avec deux
ajustements décidés ici.

1. `objectifs_precoces` somme **trois** objectifs et non quatre, `firsttower` ayant été écarté
   pour fuite.
2. `ecart_or_normalise` prend tout son sens : la phase 2 a mesuré une dérive de 7 à 10 % sur
   l'or et l'expérience entre l'entraînement et le test, que la normalisation par patch
   neutralise.

Rappel qui vaut pour toute la phase 4 : les features historiques, forme d'équipe et winrate de
champion, se calculent en fenêtre expansive sur les parties **antérieures** uniquement, triées
par date. Une moyenne globale injecterait du futur dans le passé et constituerait exactement la
fuite que cette phase vient d'éliminer.